# Tutorial 08: Amortized Neural Posterior Estimation with sbi

Simulation-based inference (SBI) with neural networks is a supervised learning approach where a network is trained to learn the mapping between simulated data and the posterior distribution of the underlying model parameters. 

The framework thus allows us to perform robust statistical inference and derive credibility intervals for parameter 
estimation with complex simulators like those developed for pulsar population synthesis. Our implementation builds on 
the [sbi](https://sbi-dev.github.io/sbi/) library ([Tejero-Cantero et al., 2020](https://arxiv.org/abs/2007.09114)).

In this tutorial, we will focus on a specific version of SBI and use the `pypopsyn/learning/train_sbi.py` script which employs so-called Neural Posterior Estimation (NPE). This allows us to train a neural density estimator on a dataset of samples of simulated neutron star populations to directly approximate the posterior distributions of the input parameters. In this case, the inference is amortized, which 
implies that the trained model is able to predict a posterior distribution for any synthetic population of neutron stars. This computation is very fast, because inference corresponds to a simple forward pass through the network.

To create the dataset of mapped simulations for this training experiment, we can either use Tutorials 04 and 05,  `04_simulation_helper_tutorial.ipynb` and `05_generator_tutorial.ipynb`, respectively, or alternatively we can take advantage of the dataset that is stored in `data/example_generator_magrot`.

To perform NPE on our dataset composed of heatmaps or 2D arrays and determine the posterior distributions of the two parameters `P_initial_log10_mean` and `B_initial_log10_mean`, we use the script `pypopsyn/learning/train_sbi.py` as follows:
```commandline
python pypopsyn/learning/train_sbi.py --configuration config_sbi.json
```
Here, the `config_sbi.json` file contains all the information required to optimize the neural network. If this `CLI` 
argument is left empty, the script will take the default `pypopsyn/learning/config_sbi.json`.

For an application of this inference approach see [Graber et al. (2024)](https://ui.adsabs.harvard.edu/abs/2024ApJ...968...16G/abstract).

In [ ]:
import collections
import corner
import json
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pathlib
import sys
import torch

from matplotlib.ticker import ScalarFormatter

import utilities.plot_settings

## Running the training script

To execute the training experiment, we run the shell command from inside this notebook.

In [ ]:
!python ../../pypopsyn/learning/train_sbi.py --configuration config_train_sbi.json

## Extracting training results

In the following, we extract the training and validation losses for the two parameters we are predicting to see how they evolve as a function of training epoch during the network optimization process.

NOTE: Make sure to change the time stamps in the variables `train_log_output_path` and `model_output_path` below to where your training results have been saved. Otherwise, the following examples will not work.

Setting the paths to the training logs and saved model.

In [ ]:
with open(f"config_train_sbi.json", "r") as read_file:
    config_train = json.load(read_file)

train_output_path = config_train["trainer"]["save_dir"]
train_log_output_path = (
    f"{train_output_path}/logs/SBI_ConvolutionMDN/20240815_193020/"
)
model_output_path = (
    f"{train_output_path}/models/SBI_ConvolutionMDN/20240815_193020/"
)

Loading the training and validation information. Note that both are contained in the `training_statistics.json` file.

In [ ]:
with open(
    f"{train_log_output_path}training_statistics.json", "r"
) as read_file:
    learning_data = json.load(read_file)

Plotting the training and validation losses as a function of training epoch for the two parameters we set out to predict.

In [ ]:
fig, ax = plt.subplots(figsize=(15, 8))

ax.plot(
    learning_data["training_log_probs"]["step"],
    learning_data["training_log_probs"]["value"],
    linestyle="-",
    linewidth=4,
    color="tab:blue",
    rasterized=True,
    label="training",
)
ax.plot(
    learning_data["validation_log_probs"]["step"],
    learning_data["validation_log_probs"]["value"],
    linestyle="-",
    linewidth=4,
    color="tab:orange",
    rasterized=True,
    label="validation",
)

ax.set_xlabel(r"Epoch")
ax.set_ylabel(r"Accuracy")
plt.legend(bbox_to_anchor=(1.05, 1), frameon=False, loc=0)

plt.show()

## Performing inference with the trained model

Once our SBI pipeline (composed of an embedding net and a neural density estimator in the above case) has been trained, it can be used to infer on an unseen dataset of generated maps and extract posterior distributions of the corresponding pulsar population parameters. The script `pypopsyn/learning/infer_sbi.py` allows us to take an experiment configuration file, a pre-trained model, and a dataset, and run the inference.

We first define the path to the trained model.

In [ ]:
trained_model_path = f"{model_output_path}trained_model.pickle"

We then run the inference script as a shell command from inside this notebook.

In [ ]:
# Construct the command.
command = f"python ../../pypopsyn/learning/infer_sbi.py --configuration config_train_sbi.json --trained_model {trained_model_path} --corner_plot True"

# Execute the command.
!{command}

## Extracting inference results

Setting the path to the inference results.

NOTE: Make sure to the time stamp in the variable `inference_results_path` below to where your inference results have been saved. Otherwise, the following example will not work.

In [ ]:
inference_output_path = config_train["infer"]["save_dir"]
inference_log_output_path = (
    f"{inference_output_path}/logs/SBI_ConvolutionMDN/20240815_193102/"
)
test_dataset_path = config_train["test_data_loader"]["dataset_path"]

Next, we load the saved posterior samples for one of our test samples to produce a corner plot.

In [ ]:
test_dataset = pd.read_csv(test_dataset_path)

In [ ]:
# Choose a specific test sample, i.e., choose an index between 0 and 3 for the 4 test samples.
sample_index = 0

# Extract the ground truth values of the parameters and the posterior samples.
ground_truth = test_dataset.iloc[sample_index][
    ["B_initial_log10_mean", "P_initial_log10_mean"]
].values
posterior_samples = torch.load(
    f"{inference_log_output_path}samples_{sample_index}.pt"
)

print("ground_truth B_initial_log10_mean: ", ground_truth[0])
print("ground_truth P_initial_log10_mean: ", ground_truth[1])

For illustration purposes, we also calculate the quantiles of the posterior samples to extract the median values for the parameters and the 95% credibility interval.

In [ ]:
quantile = np.quantile(posterior_samples, [0.025, 0.5, 0.975], axis=0)

param_median = quantile[1]
param_inf = quantile[0]
param_sup = quantile[2]

param_err_inf = param_median - param_inf
param_err_sup = param_sup - param_median

print(
    "predicted B_initial_log10_mean: ",
    f"{param_median[0]} + {param_err_sup[0]} - {param_err_inf[0]}",
)
print(
    "predicted P_initial_log10_mean: ",
    f"{param_median[1]} + {param_err_sup[1]} - {param_err_inf[1]}",
)

## Producing a corner plot

We show 2D and 1D marginalized posteriors highlighting the median in red and 99%, 95% and 68% credibility contours. 

Blue lines show the ground truths for the two parameters for our test sample.

In [ ]:
parameter_ranges = [[12, 14], [-1.5, -0.3]]
parameter_labels = [r"$\mu_{\log B}$", r"$\mu_{\log P}$"]

fig = plt.figure(figsize=(10, 10))

figure = corner.corner(
    posterior_samples.numpy(),
    bins=32,
    labels=parameter_labels,
    label_kwargs={"fontsize": 30},
    range=parameter_ranges,
    quantiles=[0.025, 0.5, 0.975],
    levels=(
        1 - np.exp(-0.5),
        1 - np.exp(-2),
        1 - np.exp(-9.0 / 2.0),
    ),  # 1, 2 and 3 sigma levels
    show_titles=True,
    title_kwargs={"fontsize": 30},
    fig=fig,
)
corner.overplot_lines(figure, param_median, color="tab:red")
corner.overplot_lines(figure, ground_truth, color="tab:blue")
corner.overplot_points(
    figure,
    param_median[None],
    marker="s",
    color="tab:red",
)
corner.overplot_points(
    figure,
    ground_truth[None],
    marker="s",
    color="tab:blue",
)

for ax in figure.get_axes():
    ax.tick_params(axis="both", labelsize=22)

## Coverage probability test

A test that can be performed to check if the trained density estimator is able to produce well calibrated posteriors is the coverage probability test. Let's consider a dataset $\{ (\boldsymbol{\theta}_i, \boldsymbol{x}_i) \}$ of simulations $\boldsymbol{x}$ that are properly labeled by the respective simulation parameters $\boldsymbol{\theta}$. We compute the posterior $\mathcal{P}(\boldsymbol{\theta}|\boldsymbol{x}_i)$ for each $\boldsymbol{x}_i$. For each posterior, we then define a credible region $\Theta_i$ with the smallest volume in the multidimensional parameter space of $\boldsymbol{\theta}$ corresponding to a total probability $1 - \alpha$ with $\alpha \in [0, 1]$:
$$
\int_{\Theta_i} \mathcal{P}(\boldsymbol{\theta}|\boldsymbol{x}_i) d \boldsymbol{\theta} = 1 - \alpha, 
$$
where $1 - \alpha$ defines the so-called credibility level.
Considering the region with the smallest volume guarantees that we are focusing on the region of the parameter space that encloses the parameter values $\boldsymbol{\theta}$ with the highest posterior probability density. In the literature, this is also called the highest posterior density region.

By counting how many $\boldsymbol{\theta}_i$ fall inside the corresponding credibility regions $\Theta_i$ we obtain a measure of how well the estimated posteriors are able to recover the label parameters $\boldsymbol{\theta}$. This count gives an estimate of the so-called coverage probability. If the posterior estimator is well calibrated, the coverage probability should be equal to the value $1 - \alpha$. If the coverage probability is higher than $1 - \alpha$, this is a symptom for a posterior estimator that tends to generate posteriors that are too conservative. On the other hand, if the coverage probability is lower than $1 - \alpha$, this indicates that the estimated posteriors are overconfident.

To assess the quality of our inference, we also load the coverage probability results. A diagonal coverage would indicate a well-calibrated posterior while while lower lines suggest overconfident and higher lines conservative posteriors.

NOTE: In this example, we only have 4 test simulations. A larger number of simulation samples would be required to properly perform a coverage test. The following is shown only for illustrative purposes.

In [ ]:
coverage_probability = np.load(
    f"{inference_log_output_path}coverage_probability.npy"
)
betas = np.linspace(0, 1, len(coverage_probability))

Plotting the coverage.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))

ax.plot(
    betas,
    coverage_probability,
    color="steelblue",
    linewidth=3,
    label="upper right",
)

ax.plot([0, 1], [0, 1], color="k", linestyle="--")
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)

ax.set_xlabel(r"Credibility level $1-\alpha$")
ax.set_ylabel(r"Coverage probability")

plt.show()

Once the model is trained and is able to produce well calibrated posteriors it can be applied on the real observed population.
To do so one needs only to specify the path to the generated maps from the ATNF catalgue in the configuration file `config_train_sbi.json` in the field related to the `test_data_loader`. Make sure also to specify the correct labels that you want to predict through the `filter_labels` option.